# Stage 2: Function-Level Cryptojacking Classifier
### Upload labeled_functions.jsonl to Drive → Train → Evaluate
**Hardware target:** Colab Free T4 (16GB VRAM)

**Pipeline:**
1. Mount Drive & copy JSONL to /content (fast local read)
2. Load labeled functions into HuggingFace Dataset
3. Fine-tune GraphCodeBERT with weighted loss
4. Evaluate + per-function inference report

**Before running:** Upload `labeled_functions.jsonl` to your Drive.


In [1]:
# Run once per Colab session
%pip install transformers datasets accelerate evaluate scikit-learn -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\cjrea\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os, torch

# ── Paths ──
LABELED_JSONL = './labeled_heuristic.jsonl'   # heuristic labeler output
STAGE1_DIR    = './gcb-assembly-final'
OUTPUT_DIR    = './gcb-stage2-checkpoints'
FINAL_DIR     = './gcb-stage2-final'

# ── RTX 3060 12GB ──
BATCH_SIZE  = 16
GRAD_ACCUM  = 1
EPOCHS      = 3
MAX_SEQ_LEN = 512
STRIDE      = 16
NUM_WORKERS = 0      # Windows: 0 to avoid multiprocessing issues in notebooks

# ── Subsampling — heuristic labels ──
MAX_MALICIOUS = 3479    # use all malicious functions (full heuristic output)
MAX_BENIGN    = 10000   # 1:3 ratio — enough contrast, tractable training time
POS_WEIGHT    = 1.0     # handled via subsampling ratio, not loss weighting

# ── Ampere optimizations ──
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device      : {device}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"JSONL       : {os.path.abspath(LABELED_JSONL)}")
print(f"Stage 1 dir : {os.path.abspath(STAGE1_DIR)}")
print(f"Malicious   : {MAX_MALICIOUS:,} | Benign: {MAX_BENIGN:,} | Ratio 1:{MAX_BENIGN//MAX_MALICIOUS}")


Device      : cuda
GPU         : NVIDIA GeForce RTX 3060
VRAM        : 12.0 GB
JSONL       : e:\Thesis\labeled_heuristic.jsonl
Stage 1 dir : e:\Thesis\gcb-assembly-final
Malicious   : 3,479 | Benign: 10,000 | Ratio 1:2


In [4]:
# Copy JSONL from Drive to local /content for fast reads
# Drive reads are slow (~20MB/s); local reads are ~500MB/s
import shutil
import json

size_mb = os.path.getsize(LABELED_JSONL) / 1024**2
print(f"JSONL size  : {size_mb:.0f} MB")

# Quick sanity check
with open(LABELED_JSONL) as f:
    first = json.loads(f.readline())
print(f"Fields      : {list(first.keys())}")
print(f"Has asm_text: {'asm_text' in first}")
print(f"Has instrs  : {'instructions' in first}")


JSONL size  : 10677 MB
Fields      : ['binary', 'binary_label', 'binary_label_name', 'function', 'entry', 'function_label', 'function_label_name', 'label_reason', 'instruction_count', 'shl_count', 'has_cpuid', 'asm_text']
Has asm_text: True
Has instrs  : False


In [5]:
import json
from datasets import Dataset, concatenate_datasets

def generate_function_data(jsonl_path):
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            if 'asm_text' in rec:
                asm_text = rec['asm_text']
            else:
                instrs = rec.get('instructions', [])
                asm_text = ' \n '.join(
                    f"{i['mnemonic']} {i['operands']}".strip()
                    for i in instrs
                )
            if not asm_text.strip():
                continue
            yield {
                'text'    : asm_text,
                'label'   : rec['function_label'],
                'function': rec.get('function', ''),
                'binary'  : rec.get('binary', ''),
            }

print("Loading dataset...")
dataset = Dataset.from_generator(
    generate_function_data,
    gen_kwargs={'jsonl_path': LABELED_JSONL}
)

total     = len(dataset)
malicious = sum(1 for x in dataset if x['label'] == 1)
benign    = total - malicious
print(f"Total    : {total:,}")
print(f"Malicious: {malicious:,} ({malicious/total*100:.2f}%)")
print(f"Benign   : {benign:,} ({benign/total*100:.2f}%)")
print(f"Ratio    : 1:{benign//malicious if malicious else 'N/A'}")

# ── Subsample: all malicious + capped benign ──────────────────
mal_data = dataset.filter(lambda x: x['label'] == 1)
ben_data = dataset.filter(lambda x: x['label'] == 0)
mal_data = mal_data.select(range(min(MAX_MALICIOUS, len(mal_data))))
ben_data = ben_data.select(range(min(MAX_BENIGN,    len(ben_data))))
dataset  = concatenate_datasets([mal_data, ben_data]).shuffle(seed=42)

print(f"\nSubsampled : {len(dataset):,} ({len(mal_data):,} mal + {len(ben_data):,} ben)")
print(f"Ratio      : 1:{len(ben_data)//len(mal_data)}")

split      = dataset.train_test_split(test_size=0.2, seed=42)
train_data = split['train']
eval_data  = split['test']
print(f"Train: {len(train_data):,} | Eval: {len(eval_data):,}")


C:\Users\cjrea\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset...


Generating train split: 4532601 examples [01:30, 50197.16 examples/s]


Total    : 4,532,601
Malicious: 3,479 (0.08%)
Benign   : 4,529,122 (99.92%)
Ratio    : 1:1301


Filter: 100%|██████████| 4532601/4532601 [00:20<00:00, 218211.30 examples/s]



Subsampled : 13,479 (3,479 mal + 10,000 ben)
Ratio      : 1:2
Train: 10,783 | Eval: 2,696


In [7]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import AutoTokenizer

print(f"Loading tokenizer from {STAGE1_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)

_worker_tokenizer = None

def tokenize_fn(examples, stage1_dir, max_seq_len, stride):
    tok = globals().get("_worker_tokenizer")
    if tok is None:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(stage1_dir)
        globals()["_worker_tokenizer"] = tok
    tokenized = tok(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=max_seq_len,
        return_overflowing_tokens=True,
        stride=stride,
    )
    sample_map = tokenized.pop('overflow_to_sample_mapping')
    tokenized['labels'] = [examples['label'][i] for i in sample_map]
    return tokenized

# Avoid multiprocessing issues on Windows when NUM_WORKERS is 0
map_num_proc = NUM_WORKERS if NUM_WORKERS and NUM_WORKERS > 0 else None

train_dataset = train_data.map(
    tokenize_fn, batched=True, batch_size=100,
    num_proc=map_num_proc,
    remove_columns=['text', 'label', 'function', 'binary'],
    desc='Tokenizing train',
    fn_kwargs={
        'stage1_dir': STAGE1_DIR,
        'max_seq_len': MAX_SEQ_LEN,
        'stride': STRIDE,
    },
)
eval_dataset = eval_data.map(
    tokenize_fn, batched=True, batch_size=100,
    num_proc=map_num_proc,
    remove_columns=['text', 'label', 'function', 'binary'],
    desc='Tokenizing eval',
    fn_kwargs={
        'stage1_dir': STAGE1_DIR,
        'max_seq_len': MAX_SEQ_LEN,
        'stride': STRIDE,
    },
)

train_dataset = train_dataset.with_format('torch')
eval_dataset  = eval_dataset.with_format('torch')

print(f"Train chunks : {len(train_dataset):,} | Eval chunks : {len(eval_dataset):,}")
print(f"Columns      : {train_dataset.column_names}")

Loading tokenizer from ./gcb-assembly-final...


Tokenizing eval: 100%|██████████| 2696/2696 [00:40<00:00, 66.24 examples/s]

Train chunks : 203,614 | Eval chunks : 48,754
Columns      : ['input_ids', 'attention_mask', 'labels']


In [10]:
import torch
import numpy as np
import evaluate
import platform
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback)

accuracy_m  = evaluate.load('accuracy')
f1_m        = evaluate.load('f1')
precision_m = evaluate.load('precision')
recall_m    = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy' : accuracy_m.compute(predictions=preds, references=labels)['accuracy'],
        'f1'       : f1_m.compute(predictions=preds, references=labels, average='binary')['f1'],
        'precision': precision_m.compute(predictions=preds, references=labels, average='binary')['precision'],
        'recall'   : recall_m.compute(predictions=preds, references=labels, average='binary')['recall'],
    }

model = AutoModelForSequenceClassification.from_pretrained(STAGE1_DIR, num_labels=2)

class WeightedTrainer(Trainer):
    """
    Handles 1:N class imbalance by upweighting the malicious class.
    Without this, the model predicts benign for everything and reports
    99%+ accuracy while missing all actual malicious functions.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weight = torch.tensor([1.0, POS_WEIGHT], device=outputs.logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weight)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

if hasattr(torch, 'compile'):
    if platform.system() != 'Windows':
        try:
            model = torch.compile(model)
            print("torch.compile() enabled")
        except Exception as exc:
            print(f"torch.compile() disabled: {exc.__class__.__name__}: {exc}")
    else:
        print("torch.compile() skipped on Windows (Triton not supported)")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.cuda.empty_cache()
print(f"Model loaded | POS_WEIGHT={POS_WEIGHT}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ./gcb-assembly-final and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.compile() skipped on Windows (Triton not supported)
Model loaded | POS_WEIGHT=1.0


In [14]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import AutoTokenizer

print(f"Loading tokenizer from {STAGE1_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)

def tokenize_fn(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_SEQ_LEN,
    )
    tokenized['labels'] = examples['label']
    return tokenized

train_dataset = train_data.map(
    tokenize_fn, batched=True, batch_size=100,
    num_proc=0,
    remove_columns=['text', 'label', 'function', 'binary'],
    desc='Tokenizing train'
)
eval_dataset = eval_data.map(
    tokenize_fn, batched=True, batch_size=100,
    num_proc=0,
    remove_columns=['text', 'label', 'function', 'binary'],
    desc='Tokenizing eval'
)

train_dataset = train_dataset.with_format('torch')
eval_dataset  = eval_dataset.with_format('torch')

print(f"Train chunks : {len(train_dataset):,} | Eval chunks : {len(eval_dataset):,}")
print(f"Columns      : {train_dataset.column_names}")

Loading tokenizer from ./gcb-assembly-final...


Tokenizing eval: 100%|██████████| 2696/2696 [00:36<00:00, 73.69 examples/s]

Train chunks : 10,783 | Eval chunks : 2,696
Columns      : ['input_ids', 'attention_mask', 'labels']


In [15]:
os.makedirs(FINAL_DIR, exist_ok=True)
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Model saved to {FINAL_DIR}")


Model saved to ./gcb-stage2-final


In [16]:
# Inference on eval set only
import numpy as np
from collections import defaultdict

inf_model = trainer.model
inf_model.to(device).eval()

all_preds  = []
all_labels = []

for batch in torch.utils.data.DataLoader(eval_dataset, batch_size=32):
    input_ids      = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels         = batch['labels']

    with torch.no_grad():
        logits = inf_model(input_ids=input_ids, attention_mask=attention_mask).logits

    preds = torch.argmax(logits, dim=-1).cpu()
    all_preds.extend(preds.tolist())
    all_labels.extend(labels.tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

tp = ((all_preds == 1) & (all_labels == 1)).sum()
tn = ((all_preds == 0) & (all_labels == 0)).sum()
fp = ((all_preds == 1) & (all_labels == 0)).sum()
fn = ((all_preds == 0) & (all_labels == 1)).sum()
total = len(all_preds)

print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print(f"Accuracy  : {(tp+tn)/total*100:.1f}%")
if tp+fp:   print(f"Precision : {tp/(tp+fp)*100:.1f}%")
if tp+fn:   print(f"Recall    : {tp/(tp+fn)*100:.1f}%")
if tp+fp and tp+fn:
    p = tp/(tp+fp); r = tp/(tp+fn)
    if p+r: print(f"F1        : {2*p*r/(p+r):.3f}")

TP=665  TN=1967  FP=18  FN=46
Accuracy  : 97.6%
Precision : 97.4%
Recall    : 93.5%
F1        : 0.954


In [19]:
import json, torch

inf_model = trainer.model
inf_model.to(device).eval()

test_files = [
    r'E:\Thesis\combined_output\calc.exe.json',
    r'E:\Thesis\combined_output\alg.exe.json',
    r'E:\Thesis\combined_output\7z2600-x64.exe.json',
]

for path in test_files:
    with open(path) as f:
        data = json.load(f)

    graphs  = data['graphs']
    label   = data['label_name']
    flagged = []

    for fn in graphs:
        instrs   = fn.get('instructions', [])
        asm_text = ' \n '.join(
            f"{i['mnemonic']} {i['operands']}".strip()
            for i in instrs
        )
        if not asm_text.strip():
            continue

        inputs = tokenizer(
            asm_text, return_tensors='pt',
            truncation=True, max_length=MAX_SEQ_LEN
        ).to(device)

        with torch.no_grad():
            logits = inf_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred  = torch.argmax(probs).item()
        conf  = probs[1].item()

        if pred == 1:
            flagged.append((fn['function'], conf))

    status = '❌ FALSE POSITIVE' if flagged else '✅ CORRECTLY BENIGN'
    print(f"\n{status} — {path.split(chr(92))[-1]} ({label})")
    for fn_name, conf in flagged:
        print(f"  Flagged: {fn_name} (conf={conf:.3f})")
    if not flagged:
        print(f"  All {len(graphs)} functions passed clean")


✅ CORRECTLY BENIGN — calc.exe.json (benign)
  All 21 functions passed clean

❌ FALSE POSITIVE — alg.exe.json (benign)
  Flagged: FUN_140004e7c (conf=0.987)
  Flagged: FUN_140005930 (conf=0.913)
  Flagged: FUN_140007114 (conf=0.863)
  Flagged: FUN_140018800 (conf=0.978)
  Flagged: FUN_14001a01c (conf=0.974)
  Flagged: FUN_14001e2d0 (conf=0.969)

❌ FALSE POSITIVE — 7z2600-x64.exe.json (benign)
  Flagged: FUN_004032e4 (conf=0.913)
  Flagged: FUN_00405a9c (conf=0.550)
  Flagged: FUN_00406f80 (conf=0.999)
  Flagged: FUN_00407b28 (conf=0.985)
  Flagged: FUN_0040860c (conf=0.881)


In [20]:
with open(r'E:\Thesis\combined_output\af421881786af65cf89b28d2a88d37658625f21f9644cf298c438267c7c92572.exe.json') as f:
    data = json.load(f)

graphs  = data['graphs']
label   = data['label_name']
flagged = []

for fn in graphs:
    instrs   = fn.get('instructions', [])
    asm_text = ' \n '.join(
        f"{i['mnemonic']} {i['operands']}".strip()
        for i in instrs
    )
    if not asm_text.strip():
        continue

    inputs = tokenizer(
        asm_text, return_tensors='pt',
        truncation=True, max_length=MAX_SEQ_LEN
    ).to(device)

    with torch.no_grad():
        logits = inf_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred  = torch.argmax(probs).item()
    conf  = probs[1].item()

    if pred == 1:
        flagged.append((fn['function'], conf))

status = '✅ DETECTED' if flagged else '❌ MISSED'
print(f"\n{status} — {label}")
for fn_name, conf in flagged:
    print(f"  Flagged: {fn_name} (conf={conf:.3f})")


✅ DETECTED — cryptojacking
  Flagged: FUN_140713141 (conf=1.000)
  Flagged: FUN_140716f9d (conf=1.000)
  Flagged: FUN_140718227 (conf=1.000)
  Flagged: FUN_140718597 (conf=1.000)
  Flagged: FUN_1407199ef (conf=1.000)
  Flagged: FUN_14071b157 (conf=0.998)
  Flagged: FUN_14071c9a6 (conf=1.000)
  Flagged: FUN_14071c9cc (conf=1.000)
  Flagged: FUN_14071cd55 (conf=0.999)
  Flagged: FUN_14071e6a6 (conf=1.000)
  Flagged: FUN_14072b50b (conf=0.999)
  Flagged: FUN_14072b543 (conf=1.000)
  Flagged: FUN_14072bdc4 (conf=0.998)
  Flagged: FUN_14072cb04 (conf=1.000)
  Flagged: FUN_14072d108 (conf=0.998)
  Flagged: FUN_14072e9f1 (conf=0.999)
  Flagged: FUN_14072ea01 (conf=1.000)
  Flagged: FUN_14072ea07 (conf=1.000)
  Flagged: FUN_14073089d (conf=1.000)
  Flagged: FUN_140731498 (conf=1.000)
  Flagged: FUN_1407359ae (conf=0.999)
  Flagged: FUN_140736031 (conf=1.000)
  Flagged: FUN_14073aa0c (conf=1.000)
  Flagged: FUN_14073e0f4 (conf=1.000)
  Flagged: FUN_14073e114 (conf=0.999)
  Flagged: FUN_14073e6